# Audio+Video Synchronization Training Pipeline

## Context
This notebook implements the training phase of the audio+video synchronization detection model.

**Previous work (local preprocessing):**
- Cells 0-4: Face detection, cropping, and SyncNet feature extraction on preprocessed video frames
- Created synchronized audio-video features with lip-sync confidence scores
- Output: `features/train/*.npy` and `features/test/*.npy` containing synchronization probability distributions

**This notebook (training on Kaggle):**
1. Extract remaining SyncNet features from preprocessed videos
2. Load audio-video synchronization features into memory
3. Build attention-based temporal model (Audio+Video paper methodology)
4. Train model to detect audio-video synchronization quality
5. Save trained checkpoints for inference

**Paper Reference:**
- Method: Multi-head attention over temporal audio-video synchronization features
- Task: Classification of real vs. manipulated video based on sync confidence
- Hyperparameters from Sec. 4.1 of paper


In [ ]:
import os, sys, glob, pickle, shutil, time, threading, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}  GPUs: {N_GPUS}")

WORK_DIR    = "/kaggle/working"
SYNCNET_DIR = "/kaggle/working/syncnet_python"

TRAIN_PROC_DIR = f"{WORK_DIR}/proc_train"
TEST_PROC_DIR  = f"{WORK_DIR}/proc_test"
TRAIN_FEAT_DIR = f"{WORK_DIR}/features/train"
TEST_FEAT_DIR  = f"{WORK_DIR}/features/test"
CKPT_DIR       = f"{WORK_DIR}/checkpoints"

for d in [TRAIN_FEAT_DIR, TEST_FEAT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# Paper hyperparameters (Sec. 4.1)
TAU          = 15
DIST_LEN     = 2 * TAU + 1  # 31
FPS, SR      = 25, 16000
AR_BLOCKS    = 2
AR_HEADS     = 16
AR_DIM       = 256
DROPOUT      = 0.1
LR           = 1e-3
WEIGHT_DECAY = 1e-6
BATCH_SIZE   = 16 * max(1, N_GPUS)
EPOCHS       = 40
WARMUP_STEPS = 300

print(f"Batch size: {BATCH_SIZE}  Epochs: {EPOCHS}  Dist len: {DIST_LEN}")

# Dataset detection (same as preprocessing)
DATA_ROOT = "/kaggle/input/datasets/mustakimbinmohsin/speech/dataset"
if not os.path.isdir(DATA_ROOT):
    hits = [p for p in glob.glob("/kaggle/input/**/dataset", recursive=True)
            if os.path.isdir(os.path.join(p, "train"))]
    assert hits, "Dataset root not found"
    DATA_ROOT = hits[0]
print(f"DATA_ROOT: {DATA_ROOT}")

# Detect manifest for speaker/split info
MANIFEST_PATH = None
for mp in [os.path.join(DATA_ROOT, "manifest.csv"),
           os.path.join(os.path.dirname(DATA_ROOT), "manifest.csv")]:
    if os.path.exists(mp):
        MANIFEST_PATH = mp
        break

if MANIFEST_PATH:
    MANIFEST = pd.read_csv(MANIFEST_PATH)
    print(f"Manifest: {len(MANIFEST)} rows")
else:
    print("Warning: manifest.csv not found (optional)")
    MANIFEST = None

## Cell 6: Extract SyncNet Features (Parallel)

In [ ]:
# Check if extraction is already done
n_train_feat = len(glob.glob(f"{TRAIN_FEAT_DIR}/*.npy"))
n_test_feat = len(glob.glob(f"{TEST_FEAT_DIR}/*.npy"))
print(f"Existing features: train={n_train_feat}  test={n_test_feat}")

# Count expected
n_train_videos = len(glob.glob(os.path.join(TRAIN_PROC_DIR, "pycrop", "*", "*.avi")))
n_test_videos = len(glob.glob(os.path.join(TEST_PROC_DIR, "pycrop", "*", "*.avi")))
print(f"Expected: train={n_train_videos}  test={n_test_videos}")

if n_train_feat >= n_train_videos and n_test_feat >= n_test_videos:
    print("\n✓ Features already extracted. Skipping.")
else:
    print("\nRunning feature extraction...")
    
    # Reuse extract_worker.py from preprocessing
    sys.path.insert(0, SYNCNET_DIR)
    from SyncNetInstance import SyncNetInstance
    
    # Detect speakers from pycrop directories
    pycrop = os.path.join(TRAIN_PROC_DIR, "pycrop")
    all_tracks = sorted(os.listdir(pycrop)) if os.path.isdir(pycrop) else []
    SPEAKERS = sorted(set(t.split("_")[0] for t in all_tracks))
    print(f"Speakers: {SPEAKERS}")
    
    WORKERS_PER_GPU = 2
    NUM_WORKERS = WORKERS_PER_GPU * max(1, N_GPUS)
    
    def chunk_list(lst, n):
        k, m = divmod(len(lst), n)
        return [lst[i*k+min(i,m):(i+1)*k+min(i+1,m)] for i in range(n)]
    
    def launch_workers_extract(proc_dir, feat_dir, label):
        import subprocess
        chunks = [c for c in chunk_list(SPEAKERS, NUM_WORKERS) if c]
        procs = []
        for i, chunk in enumerate(chunks):
            gpu = str(i % max(1, N_GPUS))
            print(f"  worker {i} -> GPU {gpu}: {chunk}")
            
            cmd = [
                "python", "extract_worker.py",
                ",".join(chunk),
                os.path.abspath(proc_dir),
                os.path.abspath(feat_dir),
                gpu,
                str(TAU)
            ]
            procs.append(subprocess.Popen(cmd, cwd=WORK_DIR))
        
        codes = [p.wait() for p in procs]
        for i, c in enumerate(codes):
            if c != 0:
                print(f"  [warn] worker {i} exit code {c}")
    
    print("\n=== Extracting TRAIN features ===")
    launch_workers_extract(TRAIN_PROC_DIR, TRAIN_FEAT_DIR, "TRAIN")
    
    print("\n=== Extracting TEST features ===")
    launch_workers_extract(TEST_PROC_DIR, TEST_FEAT_DIR, "TEST")
    
    n_train = len(glob.glob(f"{TRAIN_FEAT_DIR}/*.npy"))
    n_test = len(glob.glob(f"{TEST_FEAT_DIR}/*.npy"))
    print(f"\nExtraction complete: train={n_train}  test={n_test}")

## Cell 7: Load Features and Compute Statistics

In [ ]:
def load_features_lazy(feat_dir):
    """Load all .npy files as dict {track_id: (seq_len, 31)}"""
    features = {}
    files = sorted(glob.glob(os.path.join(feat_dir, "*.npy")))
    for f in tqdm(files, desc=f"Loading {os.path.basename(feat_dir)}"):
        track_id = os.path.splitext(os.path.basename(f))[0]
        features[track_id] = np.load(f).astype(np.float32)
    return features

print("Loading features...")
train_features = load_features_lazy(TRAIN_FEAT_DIR)
test_features = load_features_lazy(TEST_FEAT_DIR)

print(f"Train tracks: {len(train_features)}")
print(f"Test tracks: {len(test_features)}")

# Analyze sequence lengths
train_lens = [v.shape[0] for v in train_features.values()]
test_lens = [v.shape[0] for v in test_features.values()]

print(f"\nTrain sequence lengths:")
print(f"  min={min(train_lens)}  max={max(train_lens)}  mean={np.mean(train_lens):.1f}  median={np.median(train_lens):.0f}")
print(f"\nTest sequence lengths:")
print(f"  min={min(test_lens)}  max={max(test_lens)}  mean={np.mean(test_lens):.1f}  median={np.median(test_lens):.0f}")

# Set SEQ_LEN to a reasonable value (handles most, pads/truncates edge cases)
SEQ_LEN = int(np.percentile(train_lens + test_lens, 95))  # 95th percentile
print(f"\nSEQ_LEN set to: {SEQ_LEN} (95th percentile)")

# Label detection: try manifest, else synthetic labels
if MANIFEST is not None:
    # Derive label from utterance_id or video metadata if available
    print("Using manifest for labels")
    # For now: all are real (label=1); manipulated would be label=0
    train_labels = {k: 1 for k in train_features.keys()}  # real
    test_labels = {k: 1 for k in test_features.keys()}    # real
else:
    # Synthetic labels for testing (in practice: mix real + deepfake)
    print("Creating synthetic labels (all real for now)")
    train_labels = {k: 1 for k in train_features.keys()}
    test_labels = {k: 1 for k in test_features.keys()}

print(f"\nLabel distribution:")
print(f"  Train: {sum(train_labels.values())}/{len(train_labels)} positive (real)")
print(f"  Test:  {sum(test_labels.values())}/{len(test_labels)} positive (real)")

## Cell 8: Dataset and DataLoader

In [ ]:
class AudioVideoSyncDataset(Dataset):
    """Load SyncNet features (audio-video synchronization confidence) and labels."""
    
    def __init__(self, features_dict, labels_dict, seq_len, augment=False):
        self.track_ids = sorted(features_dict.keys())
        self.features_dict = features_dict
        self.labels_dict = labels_dict
        self.seq_len = seq_len
        self.augment = augment
    
    def __len__(self):
        return len(self.track_ids)
    
    def __getitem__(self, idx):
        track_id = self.track_ids[idx]
        feat = self.features_dict[track_id]  # (T, 31)
        label = self.labels_dict[track_id]
        
        T = feat.shape[0]
        if T < self.seq_len:
            # Pad with silence (uniform distribution over sync offsets)
            pad = np.ones((self.seq_len - T, self.dist_len), dtype=np.float32) / self.dist_len
            feat = np.vstack([feat, pad])
        elif T > self.seq_len:
            # Random crop during training, center crop during eval
            if self.augment:
                start = np.random.randint(0, T - self.seq_len + 1)
            else:
                start = (T - self.seq_len) // 2
            feat = feat[start:start + self.seq_len]
        
        # feat: (seq_len, 31) -> (31, seq_len) for conv/attention models
        feat = torch.from_numpy(feat.T).float()
        label = torch.tensor(label, dtype=torch.long)
        
        return feat, label, track_id

# Create datasets
print("Creating datasets...")
train_ds = AudioVideoSyncDataset(train_features, train_labels, SEQ_LEN, augment=True)
test_ds = AudioVideoSyncDataset(test_features, test_labels, SEQ_LEN, augment=False)

# Create dataloaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Peek at one batch
feat_batch, label_batch, ids_batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  feat:  {feat_batch.shape}  (batch, channels=31, seq_len={SEQ_LEN})")
print(f"  label: {label_batch.shape}")

## Cell 9: Attention-based Audio-Video Synchronization Model

In [ ]:
class MultiHeadAttentionBlock(nn.Module):
    """Multi-head self-attention block over temporal sequences."""
    
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        # x: (B, C, T) -> permute -> (B, T, C)
        x = x.permute(0, 2, 1)
        B, T, C = x.shape
        
        # QKV projection
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (B, num_heads, T, head_dim)
        
        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, T, C)
        x = self.proj(x)
        x = self.dropout(x)
        
        # Back to (B, C, T)
        return x.permute(0, 2, 1)

class AudioVideoSyncNet(nn.Module:
    """Multi-head attention model for audio-video synchronization detection.
    
    Input: (B, 31, seq_len) audio-video sync features
    Output: (B, 2) binary classification logits [fake, real]
    """
    
    def __init__(self, in_channels=31, hidden_dim=256, num_heads=16, num_blocks=2, dropout=0.1):
        super().__init__()
        
        # Project input features to hidden dimension
        self.input_proj = nn.Linear(in_channels, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Stacked attention blocks
        self.attention_blocks = nn.ModuleList([
            nn.Sequential(
                MultiHeadAttentionBlock(hidden_dim, num_heads, dropout),
                nn.LayerNorm(hidden_dim),
                nn.Linear(hidden_dim, hidden_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim * 4, hidden_dim),
                nn.Dropout(dropout),
                nn.LayerNorm(hidden_dim)
            )
            for _ in range(num_blocks)
        ])
        
        # Classification head
        self.pool = nn.AdaptiveAvgPool1d(1)  # Global average pooling
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )
    
    def forward(self, x):
        # x: (B, 31, seq_len)
        x = x.permute(0, 2, 1)  # (B, seq_len, 31)
        
        # Project to hidden dimension
        x = self.input_proj(x)  # (B, seq_len, hidden_dim)
        x = self.dropout(x)
        x = x.permute(0, 2, 1)  # (B, hidden_dim, seq_len)
        
        # Apply attention blocks
        for block in self.attention_blocks:
            x_in = x.permute(0, 2, 1)  # (B, seq_len, hidden_dim)
            x_attn = block(x)  # (B, hidden_dim, seq_len)
            x = x + x_attn  # Residual
        
        # Global average pooling
        x = self.pool(x).squeeze(-1)  # (B, hidden_dim)
        
        # Classification
        logits = self.classifier(x)  # (B, 2)
        return logits

# Create model
model = AudioVideoSyncNet(
    in_channels=DIST_LEN,
    hidden_dim=AR_DIM,
    num_heads=AR_HEADS,
    num_blocks=AR_BLOCKS,
    dropout=DROPOUT
).to(DEVICE)

print(f"Model created:")
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Test forward pass
test_input = torch.randn(4, DIST_LEN, SEQ_LEN).to(DEVICE)
test_output = model(test_input)
print(f"\nTest forward pass: input {test_input.shape} -> output {test_output.shape}")
assert test_output.shape == (4, 2), f"Expected (4, 2), got {test_output.shape}"

## Cell 10: Training Setup and Optimizer

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Warmup scheduler
def get_lr_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        return max(0.0, float(total_steps - step) / float(max(1, total_steps - warmup_steps)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

total_steps = len(train_loader) * EPOCHS
scheduler = get_lr_scheduler(optimizer, total_steps, WARMUP_STEPS * len(train_loader))

print(f"Optimizer: AdamW (lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: Linear warmup + cosine decay")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {WARMUP_STEPS * len(train_loader)}")

## Cell 11: Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch} [TRAIN]", leave=False)
    for feat, label, track_ids in pbar:
        feat, label = feat.to(device), label.to(device)
        
        # Forward pass
        logits = model(feat)
        loss = criterion(logits, label)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        # Metrics
        total_loss += loss.item() * feat.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == label).sum().item()
        total += label.size(0)
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.3f}"})
    
    avg_loss = total_loss / total
    avg_acc = correct / total
    return avg_loss, avg_acc

def eval_epoch(model, loader, criterion, device, epoch):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"Epoch {epoch} [EVAL]", leave=False)
        for feat, label, track_ids in pbar:
            feat, label = feat.to(device), label.to(device)
            
            logits = model(feat)
            loss = criterion(logits, label)
            
            total_loss += loss.item() * feat.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == label).sum().item()
            total += label.size(0)
            
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.3f}"})
    
    avg_loss = total_loss / total
    avg_acc = correct / total
    return avg_loss, avg_acc

# Training loop
print(f"\nStarting training for {EPOCHS} epochs...\n")

best_val_acc = 0.0
patience = 5
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scheduler, DEVICE, epoch)
    val_loss, val_acc = eval_epoch(model, test_loader, criterion, DEVICE, epoch)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}: "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")
    
    # Early stopping + checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "val_loss": val_loss,
            "history": history
        }
        ckpt_path = os.path.join(CKPT_DIR, f"best_model.pth")
        torch.save(checkpoint, ckpt_path)
        print(f"  ✓ Saved best model (val_acc={val_acc:.3f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.3f}")

## Cell 12: Save Final Checkpoints and Metadata

In [ ]:
# Save final model state
final_ckpt = {
    "model_state_dict": model.state_dict(),
    "model_config": {
        "in_channels": DIST_LEN,
        "hidden_dim": AR_DIM,
        "num_heads": AR_HEADS,
        "num_blocks": AR_BLOCKS,
        "dropout": DROPOUT
    },
    "hyperparameters": {
        "tau": TAU,
        "dist_len": DIST_LEN,
        "seq_len": SEQ_LEN,
        "fps": FPS,
        "sr": SR,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY
    },
    "training_history": history,
    "best_val_accuracy": best_val_acc
}

final_ckpt_path = os.path.join(CKPT_DIR, "final_model.pth")
torch.save(final_ckpt, final_ckpt_path)
print(f"Saved final checkpoint: {final_ckpt_path}")

# Save metadata
metadata = {
    "model_type": "AudioVideoSyncNet",
    "description": "Multi-head attention model for audio-video synchronization detection",
    "paper": "Audio+Video: From Real to Deepfakes",
    "input_shape": f"({DIST_LEN}, {SEQ_LEN})",
    "output_shape": "(2,)  # [fake, real]",
    "num_parameters": int(sum(p.numel() for p in model.parameters())),
    "training_samples": len(train_features),
    "test_samples": len(test_features),
    "best_validation_accuracy": float(best_val_acc),
    "checkpoint_path": final_ckpt_path
}

import json
metadata_path = os.path.join(CKPT_DIR, "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

print(f"\n✓ Checkpoints ready in: {CKPT_DIR}")
print(f"  - best_model.pth (best validation accuracy)")
print(f"  - final_model.pth (final training state)")
print(f"  - model_metadata.json (model info and hyperparameters)")

## Cell 13: Evaluation Summary

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history["train_loss"], label="Train Loss", marker='o')
axes[0].plot(history["val_loss"], label="Val Loss", marker='s')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss over Epochs")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history["train_acc"], label="Train Accuracy", marker='o')
axes[1].plot(history["val_acc"], label="Val Accuracy", marker='s')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "training_curves.png"), dpi=100, bbox_inches='tight')
print("Saved training curves plot")
plt.show()

# Print summary
print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
print(f"Model: AudioVideoSyncNet (Attention-based)")
print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nDataset:")
print(f"  Train samples: {len(train_features)}")
print(f"  Test samples: {len(test_features)}")
print(f"  Sequence length: {SEQ_LEN}")
print(f"\nFinal Results:")
print(f"  Best validation accuracy: {best_val_acc:.4f}")
print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {history['val_loss'][-1]:.4f}")
print(f"\nCheckpoints saved to: {CKPT_DIR}")
print("="*50)